# 🔍 Task 3: Automated Resume / Candidate Screening System
## Machine Learning & Natural Language Processing (NLP) for HR-Tech

### 🎯 Project Objective
Hiring teams receive hundreds of resumes for a single job opening. Manually screening each resume is time-consuming, inconsistent, and error-prone. This notebook demonstrates how to build an automated, transparent ML system that:
1. **Preprocesses & Cleans Unstructured Resume Text**
2. **Extracts Technical & Soft Skills using spaCy NLP & Custom Taxonomies**
3. **Parses & Analyzes Target Job Descriptions**
4. **Calculates TF-IDF Cosine Similarity & Weighted Skill Fit Scores**
5. **Ranks Candidates based on Role Relevance**
6. **Identifies Skill Gaps & Provides Recruiter Explanations**

---

## 1. Environment & Module Setup

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to sys.path to load local src library
sys.path.append(os.path.abspath(".."))

from src.text_cleaner import clean_text, remove_stopwords
from src.parser import DocumentParser
from src.skill_extractor import SkillExtractor
from src.matcher import ResumeMatcher

print("All system modules loaded successfully!")

## 2. Load Job Description & Candidate Resumes
We load a benchmark Senior Data Scientist Job Description and a cohort of candidate resumes representing different skill profiles (Strong fit, Web dev, DevOps, Junior Analyst, HR specialist).

In [ ]:
# Load Job Description
jd_path = os.path.join("..", "data", "sample_job_descriptions", "data_scientist.txt")
with open(jd_path, "r", encoding="utf-8") as f:
    job_description = f.read()

print("=== TARGET JOB DESCRIPTION ===")
print(job_description[:350] + "...\n")

# Load Resumes
resume_dir = os.path.join("..", "data", "sample_resumes")
resumes_list = []
for file in os.listdir(resume_dir):
    if file.endswith(".txt"):
        fpath = os.path.join(resume_dir, file)
        cname = file.replace(".txt", "").replace("_", " ").title()
        with open(fpath, "r", encoding="utf-8") as f:
            resumes_list.append({
                "id": file,
                "name": cname,
                "text": f.read()
            })

print(f"Loaded {len(resumes_list)} candidate resumes.")

## 3. Preprocessing & Skill Extraction Demo
We use spaCy Phrase Matching combined with a curated skills taxonomy to extract technical competencies.

In [ ]:
extractor = SkillExtractor(skills_db_path=os.path.join("..", "data", "skills_db.json"))

jd_skills_res = extractor.extract_skills(job_description)
print(f"Extracted {jd_skills_res['skill_count']} skills from Job Description:")
print(jd_skills_res['unique_skills'])

print("\nCategorized JD Skills:")
for category, skills in jd_skills_res['categorized_skills'].items():
    print(f"- {category}: {', '.join(skills)}")

## 4. Run Machine Learning Scoring & Ranking Pipeline
The screening engine computes:
1. **TF-IDF Cosine Similarity**: Contextual similarity between resume and JD.
2. **Skill Coverage Ratio**: Percentage of target job skills present in the resume.
3. **Mandatory Skill Score**: Weighting bonus for critical role requirements.
4. **Skill Gap Breakdown**: Matched vs Missing skills.

In [ ]:
matcher = ResumeMatcher(skill_extractor=extractor)

mandatory_skills = ["python", "machine learning", "scikit-learn", "sql"]

ranked_results = matcher.rank_candidates(
    candidates=resumes_list,
    job_description_text=job_description,
    mandatory_skills=mandatory_skills
)

# Convert results to pandas DataFrame
df_results = pd.DataFrame(ranked_results)
df_summary = df_results[["rank", "candidate_name", "fit_tier", "overall_score", "tfidf_similarity_percent", "skill_coverage_percent"]]
df_summary

## 5. Candidate Screening Leaderboard Visualization

In [ ]:
plt.figure(figsize=(10, 5))
sns.set_theme(style="whitegrid")
ax = sns.barplot(
    data=df_results,
    x="overall_score",
    y="candidate_name",
    palette="viridis"
)
plt.title("Candidate Screening & Fit Ranking Leaderboard", fontsize=14, fontweight="bold")
plt.xlabel("Overall Match Score (%)", fontsize=12)
plt.ylabel("Candidate Name", fontsize=12)

for p in ax.patches:
    width = p.get_width()
    ax.annotate(f"{width:.1f}%",
                (width + 1, p.get_y() + p.get_height() / 2.),
                ha='left', va='center', fontsize=10, color='black')

plt.xlim(0, 105)
plt.tight_layout()
plt.show()

## 6. Skill Gap & Recruiter Explanations
Non-technical recruiter explanations detailing candidate strengths, gaps, and mandatory skill fit.

In [ ]:
for cand in ranked_results:
    print("=" * 65)
    print(f"RANK #{cand['rank']}: {cand['candidate_name']} ({cand['fit_tier']} - {cand['overall_score']}%)")
    print(f"- Matched Skills ({len(cand['matched_skills'])}): {', '.join(cand['matched_skills'])}")
    print(f"- Missing Skills ({len(cand['missing_skills'])}): {', '.join(cand['missing_skills'])}")
    print("\n" + cand['explanation'] + "\n")

## 7. Key Findings & Business Impact
- **Efficiency**: Automated screening reduces recruiter review time from hours to seconds.
- **Objectivity**: Standardized TF-IDF and skill match criteria remove human bias.
- **Actionable Insights**: Skill gap identification pinpoints specific training needs or targeted interview questions.